# Colab Dense Experiments (Clean Runner)

This notebook is a fresh, minimal, runnable workflow for `task1_retrieval` dense experiments.

What it does:
- Sets up repo path (Colab or local)
- Installs only required dense dependencies
- Forces all-query run (`train + test`)
- Runs dense experiments and writes outputs to `results/task1_retrieval/english`

In [12]:
import importlib
import os
import shlex
import shutil
import subprocess
import sys
from pathlib import Path

IN_COLAB = "COLAB_RELEASE_TAG" in os.environ
DRIVE_PROJECT = Path('/content/drive/MyDrive/CLEF_2026_Humour_Project')
RUN_PROJECT = Path('/content/CLEF_2026_Humour_Project')
PERSIST_FOLDERS = ['results', 'logs']
REPO_URL = 'https://github.com/Badshah1508/Humour-Aware-Information-Retrieval-Pun-Translation-CLEF-2026-.git'


def run_cmd(cmd, cwd=None):
    cmd_str = " ".join(shlex.quote(str(part)) for part in cmd)
    print("$", cmd_str)
    completed = subprocess.run(cmd, cwd=cwd, text=True, capture_output=True)
    if completed.stdout:
        print(completed.stdout, end="")
    if completed.returncode != 0:
        if completed.stderr:
            print("\n[stderr]\n" + completed.stderr)
        raise RuntimeError(f"Command failed with exit code {completed.returncode}: {cmd_str}")


def ensure_dir_symlink(link_path: Path, target_path: Path):
    target_path.mkdir(parents=True, exist_ok=True)
    if link_path.is_symlink() or link_path.is_file():
        link_path.unlink()
    elif link_path.exists():
        shutil.rmtree(link_path)
    link_path.parent.mkdir(parents=True, exist_ok=True)
    link_path.symlink_to(target_path, target_is_directory=True)


def find_repo_root(start_dir: Path) -> Path:
    for candidate in [start_dir, *start_dir.parents]:
        if (candidate / "requirements.txt").exists() and (candidate / "src").exists():
            return candidate
    return start_dir


if IN_COLAB:
    drive = importlib.import_module("google.colab.drive")
    drive.mount('/content/drive', force_remount=False)

    if not RUN_PROJECT.exists():
        run_cmd(["git", "clone", REPO_URL, str(RUN_PROJECT)])
    else:
        run_cmd(["git", "pull"], cwd=str(RUN_PROJECT))

    for folder in PERSIST_FOLDERS:
        ensure_dir_symlink(RUN_PROJECT / folder, DRIVE_PROJECT / folder)

    REPO_DIR = RUN_PROJECT
    print('Colab mode with Drive persistence enabled.')
else:
    REPO_DIR = find_repo_root(Path.cwd())
    print('Local mode (no Drive mount).')

os.chdir(REPO_DIR)
PYTHON_EXE = sys.executable
print('Repo dir:', REPO_DIR)
print('Kernel Python:', PYTHON_EXE)

Local mode (no Drive mount).
Repo dir: c:\Users\badsh\OneDrive\Desktop\CLEF_2026_Humour_Project
Kernel Python: c:\Program Files\Python313\python.exe


In [ ]:
run_cmd([PYTHON_EXE, "-V"])
run_cmd([PYTHON_EXE, "-m", "pip", "install", "-q", "--upgrade", "pip", "setuptools", "wheel"])

if IN_COLAB:
    # Colab-stable stack
    dense_pkgs = [
        "numpy==1.26.4",
        "scipy==1.11.4",
        "scikit-learn==1.4.2",
        "pandas==2.2.2",
        "torch",
        "torchvision",
        "sentence-transformers==3.2.1",
        "transformers==4.46.3",
        "tokenizers==0.20.3",
        "safetensors==0.4.5",
        "rank-bm25==0.2.2",
        "pyyaml==6.0.2",
    ]
else:
    # Local: keep versions flexible for Python 3.13 kernels while still forcing a clean rebuild
    dense_pkgs = [
        "numpy",
        "scipy",
        "scikit-learn",
        "pandas",
        "torch",
        "torchvision",
        "sentence-transformers",
        "transformers",
        "tokenizers",
        "safetensors",
        "rank-bm25",
        "pyyaml",
    ]

run_cmd([
    PYTHON_EXE,
    "-m",
    "pip",
    "install",
    "-q",
    "--upgrade",
    "--force-reinstall",
    "--no-cache-dir",
    *dense_pkgs,
])

print("Dense dependencies installed in active kernel environment.")
print("Please restart kernel once, then rerun from Cell 1.")

$ 'c:\Program Files\Python313\python.exe' -V
Python 3.13.2
$ 'c:\Program Files\Python313\python.exe' -m pip install -q --upgrade pip setuptools wheel
$ 'c:\Program Files\Python313\python.exe' -m pip install -q --upgrade --force-reinstall --no-cache-dir numpy scipy scikit-learn pandas torch torchvision sentence-transformers transformers tokenizers safetensors rank-bm25 pyyaml


In [14]:
# Runtime knobs
os.environ["RETRIEVAL_QUERY_SPLIT"] = "all"
os.environ["DENSE_BATCH_SIZE"] = os.getenv("DENSE_BATCH_SIZE", "128")
os.environ["DENSE_EXPERIMENT_TOP_K"] = os.getenv("DENSE_EXPERIMENT_TOP_K", "100")

print("RETRIEVAL_QUERY_SPLIT=", os.environ["RETRIEVAL_QUERY_SPLIT"])
print("DENSE_BATCH_SIZE=", os.environ["DENSE_BATCH_SIZE"])
print("DENSE_EXPERIMENT_TOP_K=", os.environ["DENSE_EXPERIMENT_TOP_K"])

RETRIEVAL_QUERY_SPLIT= all
DENSE_BATCH_SIZE= 128
DENSE_EXPERIMENT_TOP_K= 100


In [15]:
import json
import os
from pathlib import Path
from typing import Dict, List

import pandas as pd

from src.data.data_loader import DataLoader
from src.evaluation.retrieval_eval import AdvancedEvaluator
from src.logger import logging
from src.retrieval.dense_retrieval import DenseRetriever
from src.retrieval.embedding_model import EmbeddingModel

TASK = "task1_retrieval"
LANGUAGE = "english"
TOP_K = int(os.getenv("DENSE_EXPERIMENT_TOP_K", "100"))
OUT_DIR = "results/task1_retrieval/english"
DENSE_BATCH_SIZE = int(os.getenv("DENSE_BATCH_SIZE", "64"))
QUERY_SPLIT = "all"


def run_dense_variant(
    corpus,
    query_texts,
    query_ids,
    model_name: str,
    normalize_embeddings: bool,
    similarity: str,
    output_file: str,
    query_prefix: str = "",
    doc_prefix: str = "",
):
    logging.info(
        f"Running dense variant model={model_name}, normalize={normalize_embeddings}, similarity={similarity}"
    )

    embedding_model = EmbeddingModel(
        model_name=model_name,
        normalize_embeddings=normalize_embeddings,
        batch_size=DENSE_BATCH_SIZE,
        query_prefix=query_prefix,
        doc_prefix=doc_prefix,
    )
    retriever = DenseRetriever(embedding_model, similarity=similarity)

    retriever.fit(corpus)
    results = retriever.search(query_texts, top_k=TOP_K, query_ids=query_ids)

    os.makedirs(OUT_DIR, exist_ok=True)
    output_path = os.path.join(OUT_DIR, output_file)

    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(results, f, indent=4)

    return output_path, results


def evaluate_results(qrels_df, results) -> Dict[str, float]:
    evaluator = AdvancedEvaluator(qrels_df, results)
    return evaluator.evaluate()


loader = DataLoader(task=TASK, language=LANGUAGE)
data = loader.load_all(query_split=QUERY_SPLIT)

corpus = data["corpus"].to_dict(orient="records")
queries_df = data["queries"]
qrels_df = data["qrels"]

query_texts = queries_df["query"].astype(str).tolist()
query_ids = queries_df["query_id"].astype(str).tolist()

# Strict all-query check
all_queries_df = loader.load_queries(split="all")
expected_all_qids = set(all_queries_df["query_id"].astype(str).tolist())
loaded_qids = set(query_ids)
if loaded_qids != expected_all_qids:
    missing = sorted(expected_all_qids - loaded_qids)
    extra = sorted(loaded_qids - expected_all_qids)
    raise ValueError(
        f"Query coverage mismatch. loaded={len(loaded_qids)} expected={len(expected_all_qids)} "
        f"missing={len(missing)} extra={len(extra)}"
    )

print(f"Running with query_split={QUERY_SPLIT} | total_queries={len(query_ids)}")
logging.info(f"Query split: {QUERY_SPLIT} | Total queries: {len(query_ids)}")

experiments: List[Dict] = [
    {
        "name": "MiniLM-cosine",
        "model_name": "sentence-transformers/all-MiniLM-L6-v2",
        "normalize_embeddings": True,
        "similarity": "cosine",
        "output_file": "dense_results_minilm_cosine.json",
    },
    {
        "name": "MPNet-cosine",
        "model_name": "sentence-transformers/all-mpnet-base-v2",
        "normalize_embeddings": True,
        "similarity": "cosine",
        "output_file": "dense_results_mpnet_cosine.json",
    },
    {
        "name": "BGE-base-dot",
        "model_name": "BAAI/bge-base-en-v1.5",
        "normalize_embeddings": True,
        "similarity": "dot",
        "output_file": "dense_results_bge_dot.json",
    },
    {
        "name": "E5-base-dot",
        "model_name": "intfloat/e5-base-v2",
        "normalize_embeddings": True,
        "similarity": "dot",
        "output_file": "dense_results_e5_dot.json",
        "query_prefix": "query: ",
        "doc_prefix": "passage: ",
    },
]

rows = []
for exp in experiments:
    output_path, results = run_dense_variant(
        corpus=corpus,
        query_texts=query_texts,
        query_ids=query_ids,
        model_name=exp["model_name"],
        normalize_embeddings=exp["normalize_embeddings"],
        similarity=exp["similarity"],
        output_file=exp["output_file"],
        query_prefix=exp.get("query_prefix", ""),
        doc_prefix=exp.get("doc_prefix", ""),
    )

    metrics = evaluate_results(qrels_df, results)
    row = {
        "Experiment": exp["name"],
        "Model": exp["model_name"],
        "Similarity": exp["similarity"],
        "Normalize": exp["normalize_embeddings"],
        "OutputFile": output_path,
    }
    row.update(metrics)
    rows.append(row)

df = pd.DataFrame(rows).sort_values(by=["MAP", "nDCG@5", "MRR"], ascending=False)
report_path = os.path.join(OUT_DIR, "dense_experiment_metrics.csv")
df.to_csv(report_path, index=False)

print("\nDense Experiment Comparison:\n")
print(df.to_string(index=False))
print(f"\nSaved experiment report: {report_path}")

# Quick artifact check
out_dir = Path(OUT_DIR)
for name in [
    "dense_experiment_metrics.csv",
    "dense_results_minilm_cosine.json",
    "dense_results_mpnet_cosine.json",
    "dense_results_bge_dot.json",
    "dense_results_e5_dot.json",
]:
    p = out_dir / name
    print(f"{name}:", "OK" if p.exists() else "MISSING")

ModuleNotFoundError: Could not import module 'PreTrainedModel'. Are this object's requirements defined correctly?